# ⚡ Quick Run Mode: Fast vs Full

This notebook supports a global run mode (EQUIPAY_MODE) for FAST/FULL runs.


In [ ]:
# GLOBAL RUN MODE (inserted)
import os
EQUIPAY_MODE = os.environ.get('EQUIPAY_MODE', 'FAST')  # FAST | FULL
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = 1000
    N_BOOTSTRAP = 100
    MAX_ITER = 200
    N_ESTIMATORS = 50
    PLOT_INLINE = False
else:
    N_SAMPLES = None
    N_BOOTSTRAP = 1000
    MAX_ITER = 1000
    N_ESTIMATORS = 200
    PLOT_INLINE = True
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; N_SAMPLES={N_SAMPLES}; N_BOOTSTRAP={N_BOOTSTRAP}")


In [ ]:
# RUN-MODE UTILITIES (safe)
import os
EQUIPAY_MODE = globals().get('EQUIPAY_MODE', os.environ.get('EQUIPAY_MODE','FAST'))

# Better defaults: 10K for FAST mode, 100K for FULL mode  
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = globals().get('N_SAMPLES', 10000)
else:
    N_SAMPLES = globals().get('N_SAMPLES', 100000)

N_BOOTSTRAP = globals().get('N_BOOTSTRAP', 100)
MAX_ITER = globals().get('MAX_ITER', 200)
N_ESTIMATORS = globals().get('N_ESTIMATORS', 50)

def enforce_fast_sample(df, n=None, seed=42):
    if EQUIPAY_MODE == 'FAST' and n is not None:
        return df.sample(n=min(len(df), n), random_state=seed)
    return df

# Conservative default: in FAST mode we skip small groups to avoid heavy per-group loops.
# In FULL mode we allow small groups to be processed for comprehensive analysis.
MIN_GROUP_N = 1000 if EQUIPAY_MODE == 'FAST' else 30
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; MIN_GROUP_N={MIN_GROUP_N}")

from src.notebook_utils import ensure_store_and_sample, get_sample_from_store, safe_weight_col
store, df_sample = ensure_store_and_sample(sample_limit=N_SAMPLES)
weight_col = safe_weight_col(df_sample)

# Data loading feedback
print(f"✓ Loaded {len(df_sample):,} records")
print(f"✓ Dataset has {len(df_sample.columns)} columns")
if 'SURVYEAR' in df_sample.columns:
    year_range = f"{df_sample['SURVYEAR'].min():.0f}-{df_sample['SURVYEAR'].max():.0f}"
    print(f"✓ Year range: {year_range}")

In [ ]:
# ============================================================================
# SETUP AND IMPORTS
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg

from data_store import EquiPayDataStore, Agg, Func
from src.leakage_prevention import LeakageGuard

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Imports successful (6-Layer Data Store Architecture)")

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Quantile regression
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg

# Project imports
from src.constants import COLS, normalize_column_names

# Gap analysis modules
from src.gap_analysis import (
    QuantileGapAnalyzer,
    glass_ceiling_test,
    sticky_floor_test,
)
from src.gap_analysis.core import weighted_quantile, weighted_mean

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print("✓ Libraries loaded successfully")
print("✓ Quantile regression modules loaded")

In [ ]:
# ============================================================================
# DATA LOADING VIA EquiPayDataStore (6-Layer Architecture)
# ============================================================================

# Initialize data store with new architecture
store = EquiPayDataStore(
    parquet_path=str(PROJECT_ROOT / "data" / "parquet"),
    memory_limit_mb=1000,
    enable_cache=True
)

# Load data with valid wages using new API
try:
    df = store.query().where('HRLYEARN > 0').to_pandas()
except Exception:
    df = get_sample_from_store(query='HRLYEARN > 0', limit=N_SAMPLES)

df = normalize_column_names(df)

# Create analysis variables
df['IS_FEMALE'] = (df['GENDER'] == 2).astype(int)
df['LOG_WAGE'] = np.log(df['REAL_HRLYEARN'].clip(lower=1))

# Get summary stats
total_records = store.count()
years = store.years()

print(f"✓ Loaded {len(df):,} records (6-Layer Architecture)")
print(f"✓ Years: {min(years)} - {max(years)}")

In [ ]:
# Vectorized CPI deflation (replace row-wise apply if present)
try:
    cpi_dict
except NameError:
    from src.macro_data import get_macro_dataframe
    cpi_dict = get_macro_dataframe().set_index('year')['cpi'].to_dict()

df['REAL_HRLYEARN_vec'] = df['HRLYEARN'] / df['YEAR'].map(cpi_dict).fillna(1.0)
# Parity check with pre-existing REAL_HRLYEARN if present
if 'REAL_HRLYEARN' in df.columns:
    mask = df['REAL_HRLYEARN'].notna() | df['REAL_HRLYEARN_vec'].notna()
    assert np.allclose(df.loc[mask,'REAL_HRLYEARN'].fillna(0).values,
                       df.loc[mask,'REAL_HRLYEARN_vec'].fillna(0).values, rtol=1e-6, equal_nan=True), 'Vectorized deflation does not match existing REAL_HRLYEARN'
    df['REAL_HRLYEARN'] = df['REAL_HRLYEARN_vec']
    print('✓ REAL_HRLYEARN parity ok and vectorized column used')
else:
    df['REAL_HRLYEARN'] = df['REAL_HRLYEARN_vec']
    print('✓ REAL_HRLYEARN created via vectorized deflation')

## 1. Unconditional Quantile Analysis

First, examine the raw gender gap at different points of the wage distribution.

In [ ]:
# ============================================================================
# UNCONDITIONAL QUANTILE GAPS
# ============================================================================

print("=" * 70)
print("UNCONDITIONAL QUANTILE GAPS (Weighted)")
print("=" * 70)

quantiles = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]

male_df = df[df['GENDER'] == 1]
female_df = df[df['GENDER'] == 2]

results = []
for q in quantiles:
    male_q = weighted_quantile(
        male_df['REAL_HRLYEARN'].values,
        male_df['FINALWT'].values,
        q
    )
    female_q = weighted_quantile(
        female_df['REAL_HRLYEARN'].values,
        female_df['FINALWT'].values,
        q
    )
    gap_abs = female_q - male_q
    gap_pct = (female_q - male_q) / male_q * 100
    
    results.append({
        'Quantile': f'{q:.0%}',
        'τ': q,
        'Male': male_q,
        'Female': female_q,
        'Gap ($)': gap_abs,
        'Gap (%)': gap_pct
    })

quantile_df = pd.DataFrame(results)
print("\n" + quantile_df.to_string(index=False))

# Test for glass ceiling
gap_10 = quantile_df[quantile_df['τ'] == 0.10]['Gap (%)'].values[0]
gap_50 = quantile_df[quantile_df['τ'] == 0.50]['Gap (%)'].values[0]
gap_90 = quantile_df[quantile_df['τ'] == 0.90]['Gap (%)'].values[0]

print("\n" + "=" * 70)
print("GLASS CEILING / STICKY FLOOR ASSESSMENT")
print("=" * 70)
print(f"\nGap at 10th percentile: {gap_10:.1f}%")
print(f"Gap at 50th percentile: {gap_50:.1f}%")
print(f"Gap at 90th percentile: {gap_90:.1f}%")

if abs(gap_90) > abs(gap_50) > abs(gap_10):
    print("\n🔴 GLASS CEILING: Gap widens at top of distribution")
elif abs(gap_10) > abs(gap_50) > abs(gap_90):
    print("\n🔵 STICKY FLOOR: Gap largest at bottom of distribution")
else:
    print("\n⚪ MIXED PATTERN: No clear ceiling or floor effect")

In [ ]:
# ============================================================================
# VISUALIZATION: Quantile Gap Profile
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Absolute wages by quantile
ax1 = axes[0]
x = np.arange(len(quantiles))
width = 0.35

bars1 = ax1.bar(x - width/2, quantile_df['Male'], width, label='Male', color='#1f77b4')
bars2 = ax1.bar(x + width/2, quantile_df['Female'], width, label='Female', color='#e377c2')

ax1.set_xlabel('Wage Quantile')
ax1.set_ylabel('Hourly Wage (2010$)')
ax1.set_title('Wage Distribution by Gender', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(quantile_df['Quantile'])
ax1.legend()

# Right: Gap across quantiles
ax2 = axes[1]
ax2.fill_between(quantile_df['τ'], 0, quantile_df['Gap (%)'], 
                  alpha=0.3, color='#d62728')
ax2.plot(quantile_df['τ'], quantile_df['Gap (%)'], 'o-', 
         color='#d62728', linewidth=2, markersize=8)

ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.axhline(y=gap_50, color='gray', linestyle='--', 
            label=f'Median gap: {gap_50:.1f}%')

ax2.set_xlabel('Quantile (τ)')
ax2.set_ylabel('Gender Gap (%)')
ax2.set_title('Gender Gap Across Wage Distribution', fontweight='bold')
ax2.legend()

# Add annotations
ax2.annotate('Low earners', xy=(0.10, gap_10), xytext=(0.15, gap_10-3),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
ax2.annotate('High earners\n(Glass Ceiling?)', xy=(0.90, gap_90), 
             xytext=(0.75, gap_90+5),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/glass_ceiling_quantile_profile.png', 
            dpi=300, bbox_inches='tight')
plt.show()

## 2. Conditional Quantile Regression

Estimate the gender coefficient at each quantile, controlling for human capital.

In [ ]:
# ============================================================================
# CONDITIONAL QUANTILE REGRESSION
# ============================================================================

print("=" * 70)
print("CONDITIONAL QUANTILE REGRESSION")
print("Model: ln(Wage) = α(τ) + β(τ)Female + γ(τ)Controls + ε(τ)")
print("=" * 70)

# Prepare data for regression
df_qr = df[['LOG_WAGE', 'IS_FEMALE', 'EDUC', 'AGE_12', 'TENURE', 
            'NOC_10', 'PROV', 'FINALWT']].dropna()

# Sample for computational efficiency
n_sample = min(100_000, len(df_qr))
df_qr_sample = df_qr.sample(n=n_sample, random_state=42)

# Features (excluding occupation and province dummies for simplicity)
X = df_qr_sample[['IS_FEMALE', 'EDUC', 'AGE_12', 'TENURE']]
X = sm.add_constant(X)
y = df_qr_sample['LOG_WAGE']

# Quantiles to estimate
quantiles_qr = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]

qr_results = []
print(f"\nEstimating quantile regressions at τ = {quantiles_qr}")
print("(This may take a few minutes...)\n")

for q in quantiles_qr:
    model = QuantReg(y, X)
    result = model.fit(q=q, max_iter=MAX_ITER)
    
    # Extract female coefficient
    coef = result.params['IS_FEMALE']
    se = result.bse['IS_FEMALE']
    pval = result.pvalues['IS_FEMALE']
    
    # Convert to percentage
    pct_effect = (np.exp(coef) - 1) * 100
    
    qr_results.append({
        'τ': q,
        'Coef': coef,
        'SE': se,
        'p-value': pval,
        'Effect (%)': pct_effect,
        'CI_lower': (np.exp(coef - 1.96*se) - 1) * 100,
        'CI_upper': (np.exp(coef + 1.96*se) - 1) * 100
    })
    
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    print(f"τ = {q:.2f}: Female = {pct_effect:+.1f}% [{(np.exp(coef - 1.96*se) - 1) * 100:+.1f}%, {(np.exp(coef + 1.96*se) - 1) * 100:+.1f}%] {sig}")

qr_df = pd.DataFrame(qr_results)

In [ ]:
# ============================================================================
# VISUALIZATION: Quantile Regression Coefficients
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Plot coefficient with CI
ax.fill_between(qr_df['τ'], qr_df['CI_lower'], qr_df['CI_upper'],
                alpha=0.3, color='#d62728', label='95% CI')
ax.plot(qr_df['τ'], qr_df['Effect (%)'], 'o-', color='#d62728', 
        linewidth=2, markersize=8, label='Female coefficient')

# Reference line
ax.axhline(y=0, color='black', linewidth=0.8)

# OLS comparison (mean effect)
ols = sm.OLS(y, X).fit()
ols_effect = (np.exp(ols.params['IS_FEMALE']) - 1) * 100
ax.axhline(y=ols_effect, color='gray', linestyle='--', 
           label=f'OLS (mean): {ols_effect:.1f}%')

ax.set_xlabel('Quantile (τ)', fontsize=12)
ax.set_ylabel('Gender Effect (%)', fontsize=12)
ax.set_title('Gender Wage Gap Across the Distribution\n(Conditional Quantile Regression)',
             fontweight='bold', fontsize=13)
ax.legend(loc='lower left')

# Annotations
bottom_gap = qr_df[qr_df['τ'] == 0.10]['Effect (%)'].values[0]
top_gap = qr_df[qr_df['τ'] == 0.90]['Effect (%)'].values[0]

ax.annotate('', xy=(0.90, top_gap), xytext=(0.50, qr_df[qr_df['τ'] == 0.50]['Effect (%)'].values[0]),
            arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.5))

# Add glass ceiling indicator
if abs(top_gap) > abs(bottom_gap):
    ax.text(0.85, top_gap - 3, 'Glass Ceiling', fontsize=10, color='#d62728', 
            fontweight='bold', ha='center')

plt.tight_layout()
plt.savefig('../reports/figures/quantile_regression_coefficients.png', 
            dpi=300, bbox_inches='tight')
plt.show()

## 3. Glass Ceiling Index

Formal test following Albrecht et al. (2003).

In [ ]:
# ============================================================================
# GLASS CEILING INDEX (Albrecht et al. 2003)
# ============================================================================

print("=" * 70)
print("GLASS CEILING INDEX (GCI)")
print("=" * 70)
print("""
The Glass Ceiling Index compares gaps at the top vs middle of the distribution:

GCI = |Gap(τ=0.90)| / |Gap(τ=0.50)|

Interpretation:
- GCI > 1: Glass ceiling effect (gap widens at top)
- GCI ≈ 1: Uniform gap across distribution
- GCI < 1: Gap narrows at top (reverse ceiling)
""")

# Calculate GCI using conditional gaps
gap_50_cond = abs(qr_df[qr_df['τ'] == 0.50]['Effect (%)'].values[0])
gap_90_cond = abs(qr_df[qr_df['τ'] == 0.90]['Effect (%)'].values[0])

gci = gap_90_cond / gap_50_cond if gap_50_cond != 0 else np.nan

print(f"\nResults:")
print(f"  Gap at τ=0.50 (median):     {qr_df[qr_df['τ'] == 0.50]['Effect (%)'].values[0]:.1f}%")
print(f"  Gap at τ=0.90 (top):        {qr_df[qr_df['τ'] == 0.90]['Effect (%)'].values[0]:.1f}%")
print(f"  Gap at τ=0.95 (very top):   {qr_df[qr_df['τ'] == 0.95]['Effect (%)'].values[0]:.1f}%")
print(f"\n  Glass Ceiling Index: {gci:.2f}")

if gci > 1.2:
    print(f"\n  🔴 STRONG GLASS CEILING: Gap {(gci-1)*100:.0f}% larger at top")
elif gci > 1.0:
    print(f"\n  🟡 MODERATE GLASS CEILING: Gap {(gci-1)*100:.0f}% larger at top")
elif gci < 0.8:
    print(f"\n  🔵 STICKY FLOOR: Gap larger at bottom than top")
else:
    print(f"\n  ⚪ UNIFORM GAP: Similar across distribution")

In [ ]:
# ============================================================================
# STATISTICAL TEST FOR GLASS CEILING
# ============================================================================

print("\n" + "=" * 70)
print("FORMAL TEST: H0: Gap(0.90) = Gap(0.50)")
print("=" * 70)

# Bootstrap test
n_bootstrap = N_BOOTSTRAP
gap_diffs = []

print(f"\nRunning {n_bootstrap} bootstrap iterations...")

np.random.seed(42)
for i in range(n_bootstrap):
    # Resample
    idx = np.random.choice(len(df_qr_sample), size=len(df_qr_sample), replace=True)
    X_boot = X.iloc[idx]
    y_boot = y.iloc[idx]
    
    try:
        # Fit at 0.50 and 0.90
        qr_50 = QuantReg(y_boot, X_boot).fit(q=0.50, max_iter=MAX_ITER)
        qr_90 = QuantReg(y_boot, X_boot).fit(q=0.90, max_iter=MAX_ITER)
        
        coef_50 = qr_50.params['IS_FEMALE']
        coef_90 = qr_90.params['IS_FEMALE']
        
        gap_diffs.append(coef_90 - coef_50)
    except Exception:
        continue

gap_diffs = np.array(gap_diffs)

# Test statistic
observed_diff = (qr_df[qr_df['τ'] == 0.90]['Coef'].values[0] - 
                 qr_df[qr_df['τ'] == 0.50]['Coef'].values[0])
se_diff = np.std(gap_diffs)
z_stat = observed_diff / se_diff if se_diff > 0 else np.nan
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f"\nDifference in gaps (0.90 - 0.50): {observed_diff:.4f}")
print(f"Bootstrap SE: {se_diff:.4f}")
print(f"z-statistic: {z_stat:.2f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print(f"\n✗ Reject H0: Gap at 90th percentile significantly different from median")
    if observed_diff < 0:
        print(f"  → Glass ceiling confirmed (gap more negative at top)")
    else:
        print(f"  → Reverse ceiling (gap less negative at top)")
else:
    print(f"\n✓ Fail to reject H0: No significant difference between 90th and median gap")

## 4. Glass Ceiling by Occupation

Does the glass ceiling effect vary by occupation?

In [ ]:
# ============================================================================
# GLASS CEILING BY OCCUPATION
# ============================================================================

from src.constants import NOC_10_LABELS

print("=" * 70)
print("GLASS CEILING INDEX BY OCCUPATION")
print("=" * 70)

gci_by_occ = []

for occ in sorted(df_qr['NOC_10'].dropna().unique()):
    occ_df = df_qr[df_qr['NOC_10'] == occ]
    if len(occ_df) < MIN_GROUP_N:
        continue
    
    # Calculate unconditional gaps
    male_df = occ_df[occ_df['IS_FEMALE'] == 0]
    female_df = occ_df[occ_df['IS_FEMALE'] == 1]
    
    min_subgroup_n = max(30, MIN_GROUP_N // 10)
    if len(male_df) < min_subgroup_n or len(female_df) < min_subgroup_n:
        continue
    
    # Weighted quantiles
    m_50 = weighted_quantile(male_df['LOG_WAGE'].values, male_df['FINALWT'].values, 0.50)
    f_50 = weighted_quantile(female_df['LOG_WAGE'].values, female_df['FINALWT'].values, 0.50)
    m_90 = weighted_quantile(male_df['LOG_WAGE'].values, male_df['FINALWT'].values, 0.90)
    f_90 = weighted_quantile(female_df['LOG_WAGE'].values, female_df['FINALWT'].values, 0.90)
    
    gap_50 = (np.exp(f_50 - m_50) - 1) * 100
    gap_90 = (np.exp(f_90 - m_90) - 1) * 100
    
    gci = abs(gap_90) / abs(gap_50) if abs(gap_50) > 0.5 else np.nan
    
    gci_by_occ.append({
        'Occupation': NOC_10_LABELS.get(occ, str(occ)),
        'NOC_10': occ,
        'Gap @ 50%': gap_50,
        'Gap @ 90%': gap_90,
        'GCI': gci,
        'N': len(occ_df),
        'Effect': 'Ceiling' if gci > 1.1 else 'Floor' if gci < 0.9 else 'Uniform'
    })

gci_occ_df = pd.DataFrame(gci_by_occ).sort_values('GCI', ascending=False)
print(gci_occ_df.to_string(index=False))

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#d62728' if e == 'Ceiling' else '#2ca02c' if e == 'Floor' else '#7f7f7f' 
          for e in gci_occ_df['Effect']]

bars = ax.barh(gci_occ_df['Occupation'], gci_occ_df['GCI'], color=colors)
ax.axvline(x=1.0, color='black', linewidth=1.5, linestyle='-', label='No ceiling/floor')

ax.set_xlabel('Glass Ceiling Index (GCI)')
ax.set_title('Glass Ceiling Index by Occupation\n(GCI > 1 = Ceiling, GCI < 1 = Floor)',
             fontweight='bold')

# Add value labels
for bar, val in zip(bars, gci_occ_df['GCI']):
    if not np.isnan(val):
        ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/gci_by_occupation.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Time Trend of Glass Ceiling

Has the glass ceiling effect changed over time?

In [ ]:
# ============================================================================
# GLASS CEILING TREND OVER TIME
# ============================================================================

print("=" * 70)
print("GLASS CEILING INDEX TREND (2010-2024)")
print("=" * 70)

gci_trend = []

for year in sorted(df['SURVYEAR'].unique()):
    year_df = df[df['SURVYEAR'] == year]
    
    male_df = year_df[year_df['GENDER'] == 1]
    female_df = year_df[year_df['GENDER'] == 2]
    
    # Weighted quantiles
    m_50 = weighted_quantile(male_df['REAL_HRLYEARN'].values, male_df['FINALWT'].values, 0.50)
    f_50 = weighted_quantile(female_df['REAL_HRLYEARN'].values, female_df['FINALWT'].values, 0.50)
    m_90 = weighted_quantile(male_df['REAL_HRLYEARN'].values, male_df['FINALWT'].values, 0.90)
    f_90 = weighted_quantile(female_df['REAL_HRLYEARN'].values, female_df['FINALWT'].values, 0.90)
    
    gap_50 = (f_50 - m_50) / m_50 * 100
    gap_90 = (f_90 - m_90) / m_90 * 100
    
    gci = abs(gap_90) / abs(gap_50) if abs(gap_50) > 0.5 else np.nan
    
    gci_trend.append({
        'Year': year,
        'Gap @ 50%': gap_50,
        'Gap @ 90%': gap_90,
        'GCI': gci
    })

gci_trend_df = pd.DataFrame(gci_trend)
print(gci_trend_df.to_string(index=False))

In [ ]:
# Visualization: Trend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Gap at different quantiles over time
ax1 = axes[0]
ax1.plot(gci_trend_df['Year'], gci_trend_df['Gap @ 50%'], 'o-', 
         label='Median (τ=0.50)', color='#1f77b4', linewidth=2)
ax1.plot(gci_trend_df['Year'], gci_trend_df['Gap @ 90%'], 's-', 
         label='Top (τ=0.90)', color='#d62728', linewidth=2)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Year')
ax1.set_ylabel('Gender Gap (%)')
ax1.set_title('Gender Gap at Median vs Top Over Time', fontweight='bold')
ax1.legend()

# Right: GCI over time
ax2 = axes[1]
ax2.plot(gci_trend_df['Year'], gci_trend_df['GCI'], 'o-', 
         color='#9467bd', linewidth=2, markersize=8)
ax2.axhline(y=1.0, color='black', linewidth=1, linestyle='--', label='No ceiling')
ax2.fill_between(gci_trend_df['Year'], 1.0, gci_trend_df['GCI'], 
                  where=gci_trend_df['GCI'] > 1.0, alpha=0.3, color='#d62728',
                  label='Ceiling effect')
ax2.set_xlabel('Year')
ax2.set_ylabel('Glass Ceiling Index')
ax2.set_title('Glass Ceiling Index Over Time', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('../reports/figures/gci_trend.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Summary and Key Findings

In [ ]:
# ============================================================================
# SUMMARY
# ============================================================================

print("=" * 70)
print("GLASS CEILING ANALYSIS: KEY FINDINGS")
print("=" * 70)

print(f"""
1. UNCONDITIONAL ANALYSIS:
   • Gap at 10th percentile: {gap_10:.1f}%
   • Gap at 50th percentile: {gap_50:.1f}%
   • Gap at 90th percentile: {gap_90:.1f}%
   
2. CONDITIONAL ANALYSIS (controlling for human capital):
   • Gap at median: {qr_df[qr_df['τ'] == 0.50]['Effect (%)'].values[0]:.1f}%
   • Gap at 90th: {qr_df[qr_df['τ'] == 0.90]['Effect (%)'].values[0]:.1f}%
   • Glass Ceiling Index: {gci:.2f}
   
3. VARIATION BY OCCUPATION:
   • Strongest ceiling: {gci_occ_df.iloc[0]['Occupation']} (GCI = {gci_occ_df.iloc[0]['GCI']:.2f})
   • Weakest ceiling: {gci_occ_df.iloc[-1]['Occupation']} (GCI = {gci_occ_df.iloc[-1]['GCI']:.2f})
   
4. TIME TREND:
   • GCI in {gci_trend_df['Year'].min()}: {gci_trend_df.iloc[0]['GCI']:.2f}
   • GCI in {gci_trend_df['Year'].max()}: {gci_trend_df.iloc[-1]['GCI']:.2f}
   • Trend: {'Increasing' if gci_trend_df.iloc[-1]['GCI'] > gci_trend_df.iloc[0]['GCI'] else 'Decreasing'} glass ceiling

5. POLICY IMPLICATIONS:
   • Glass ceiling effects suggest barriers to women's advancement to top positions
   • Within-occupation ceiling effects point to firm-level promotion disparities
   • Targeted interventions needed at senior levels
""")

# Save results
quantile_df.to_csv('../reports/quantile_gaps.csv', index=False)
qr_df.to_csv('../reports/quantile_regression_results.csv', index=False)
gci_occ_df.to_csv('../reports/gci_by_occupation.csv', index=False)
gci_trend_df.to_csv('../reports/gci_trend.csv', index=False)

print("\n✓ Results saved to reports/")

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("Loading data from DuckDB...")

store = LFSDataStore()

df = store.query("""
    SELECT 
        YEAR, PROV,
        SEX, AGE, EDUC, IMMIG,
        NOC_10, NAICS_21, UNION, FTPTMAIN,
        HRLYEARN, FINALWT,
        CASE WHEN SEX = 2 THEN 1 ELSE 0 END AS IS_FEMALE
    FROM lfs_data
    WHERE HRLYEARN > 0 AND HRLYEARN < 500
      AND LFSSTAT IN (1, 2)
      AND AGE BETWEEN 25 AND 54  -- Prime working age for ceiling analysis
""")

# CPI adjustment
cpi = {2010: 0.85, 2011: 0.87, 2012: 0.89, 2013: 0.90, 2014: 0.92,
       2015: 0.93, 2016: 0.94, 2017: 0.96, 2018: 0.98, 2019: 1.00,
       2020: 1.00, 2021: 1.03, 2022: 1.10, 2023: 1.14, 2024: 1.17, 2025: 1.20}
df['REAL_HRLYEARN'] = df.apply(lambda x: x['HRLYEARN'] / cpi.get(x['YEAR'], 1.0), axis=1)
df['LOG_WAGE'] = np.log(df['REAL_HRLYEARN'])

# Create control variables
df['AGE_SQ'] = df['AGE'] ** 2
df['EXP_PROXY'] = df['AGE'] - df['EDUC'] - 6
df['EXP_PROXY'] = df['EXP_PROXY'].clip(lower=0)

print(f"\n📊 Sample: {len(df):,} observations")
print(f"   Age range: 25-54 (prime working age)")
print(f"   Female share: {df['IS_FEMALE'].mean():.1%}")

## 1. Wage Distribution by Gender

In [ ]:
# ============================================================================
# WAGE DISTRIBUTIONS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of wages
ax1 = axes[0]
male_wages = df[df['IS_FEMALE'] == 0]['REAL_HRLYEARN']
female_wages = df[df['IS_FEMALE'] == 1]['REAL_HRLYEARN']

ax1.hist(male_wages, bins=50, alpha=0.5, label='Male', color='steelblue', density=True)
ax1.hist(female_wages, bins=50, alpha=0.5, label='Female', color='coral', density=True)
ax1.axvline(male_wages.median(), color='steelblue', linestyle='--', linewidth=2)
ax1.axvline(female_wages.median(), color='coral', linestyle='--', linewidth=2)
ax1.set_xlabel('Real Hourly Wage ($)')
ax1.set_ylabel('Density')
ax1.set_title('Wage Distribution by Gender')
ax1.legend()
ax1.set_xlim(0, 80)

# Quantile comparison
ax2 = axes[1]
quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]
male_q = [male_wages.quantile(q) for q in quantiles]
female_q = [female_wages.quantile(q) for q in quantiles]

x = np.arange(len(quantiles))
width = 0.35
ax2.bar(x - width/2, male_q, width, label='Male', color='steelblue')
ax2.bar(x + width/2, female_q, width, label='Female', color='coral')
ax2.set_xticks(x)
ax2.set_xticklabels(['10th', '25th', '50th', '75th', '90th'])
ax2.set_xlabel('Percentile')
ax2.set_ylabel('Hourly Wage ($)')
ax2.set_title('Wage by Quantile and Gender')
ax2.legend()

plt.tight_layout()
plt.show()

# Calculate raw gaps at each quantile
print("\n" + "=" * 60)
print("RAW GENDER GAP BY QUANTILE")
print("=" * 60)
print(f"\n{'Quantile':<12} {'Male':>10} {'Female':>10} {'Gap':>10} {'Gap %':>10}")
print("-" * 55)

for i, q in enumerate(quantiles):
    gap = female_q[i] - male_q[i]
    gap_pct = gap / male_q[i]
    print(f"{q*100:.0f}th{'':<7} ${male_q[i]:>8.2f} ${female_q[i]:>8.2f} ${gap:>8.2f} {gap_pct:>9.1%}")

## 2. Quantile Regression Analysis

In [ ]:
# ============================================================================
# QUANTILE REGRESSION
# ============================================================================

# Prepare data (sample for speed)
df_sample = df.dropna(subset=['LOG_WAGE', 'IS_FEMALE', 'AGE', 'EDUC', 'EXP_PROXY']).copy()

if len(df_sample) > 500000:
    df_sample = df_sample.sample(n=500000, random_state=42)

# Independent variables
X = df_sample[['IS_FEMALE', 'AGE', 'AGE_SQ', 'EDUC', 'EXP_PROXY']].copy()
X = sm.add_constant(X)
y = df_sample['LOG_WAGE']

# Quantiles to estimate
quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]

print("Running quantile regressions...")
print("(This may take a few minutes)\n")

qreg_results = []

for q in quantiles:
    print(f"  Estimating {q*100:.0f}th percentile...", end=" ")
    model = QuantReg(y, X)
    result = model.fit(q=q, max_iter=MAX_ITER)
    
    qreg_results.append({
        'quantile': q,
        'female_coef': result.params['IS_FEMALE'],
        'female_se': result.bse['IS_FEMALE'],
        'female_pvalue': result.pvalues['IS_FEMALE'],
        'pseudo_r2': result.prsquared
    })
    print(f"β = {result.params['IS_FEMALE']:.4f}")

qreg_df = pd.DataFrame(qreg_results)

# Also run OLS for comparison
ols_model = sm.OLS(y, X).fit()
ols_female = ols_model.params['IS_FEMALE']

print(f"\n  OLS (mean):               β = {ols_female:.4f}")

In [ ]:
# ============================================================================
# VISUALIZATION: QUANTILE REGRESSION RESULTS
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Plot quantile coefficients with CI
q_vals = qreg_df['quantile'] * 100
coefs = qreg_df['female_coef']
ci_lower = coefs - 1.96 * qreg_df['female_se']
ci_upper = coefs + 1.96 * qreg_df['female_se']

ax.fill_between(q_vals, ci_lower, ci_upper, alpha=0.3, color='coral')
ax.plot(q_vals, coefs, 'o-', color='coral', linewidth=2, markersize=10, label='Quantile Regression')

# OLS line
ax.axhline(y=ols_female, color='steelblue', linestyle='--', linewidth=2, label=f'OLS Mean ({ols_female:.3f})')

# Zero line
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

ax.set_xlabel('Quantile', fontsize=12)
ax.set_ylabel('Female Coefficient (log wage)', fontsize=12)
ax.set_title('Gender Wage Gap Across the Distribution\n(Quantile Regression Estimates)', fontsize=14)
ax.legend(loc='upper right')

# Add annotations
for i, row in qreg_df.iterrows():
    ax.annotate(f'{row["female_coef"]:.3f}', 
                (row['quantile']*100, row['female_coef']),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=9)

# Indicate glass ceiling / sticky floor
if qreg_df.iloc[-1]['female_coef'] < qreg_df.iloc[0]['female_coef']:
    ax.annotate('Glass Ceiling', xy=(90, qreg_df.iloc[-1]['female_coef']), 
                xytext=(80, qreg_df.iloc[-1]['female_coef'] - 0.03),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/quantile_regression_gaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Glass Ceiling Index

In [ ]:
# ============================================================================
# GLASS CEILING INDEX
# ============================================================================

# Glass Ceiling Index = Gap at 90th / Gap at 10th
# If > 1: Glass ceiling (gap widens at top)
# If < 1: Sticky floor (gap widest at bottom)

gap_10th = abs(qreg_df[qreg_df['quantile'] == 0.10]['female_coef'].values[0])
gap_50th = abs(qreg_df[qreg_df['quantile'] == 0.50]['female_coef'].values[0])
gap_90th = abs(qreg_df[qreg_df['quantile'] == 0.90]['female_coef'].values[0])

glass_ceiling_index = gap_90th / gap_10th if gap_10th > 0 else np.nan
ceiling_vs_median = gap_90th / gap_50th if gap_50th > 0 else np.nan

print("=" * 70)
print("GLASS CEILING / STICKY FLOOR ANALYSIS")
print("=" * 70)

print("\n--- Quantile Regression Gender Coefficients ---")
print(f"   10th percentile gap: {gap_10th:.4f} ({gap_10th*100:.1f}% penalty)")
print(f"   50th percentile gap: {gap_50th:.4f} ({gap_50th*100:.1f}% penalty)")
print(f"   90th percentile gap: {gap_90th:.4f} ({gap_90th*100:.1f}% penalty)")

print("\n--- Glass Ceiling Index ---")
print(f"   GCI (90th/10th): {glass_ceiling_index:.3f}")
print(f"   GCI (90th/50th): {ceiling_vs_median:.3f}")

print("\n--- Interpretation ---")
if glass_ceiling_index > 1.1:
    print(f"   🔴 GLASS CEILING DETECTED")
    print(f"      The gender gap at the 90th percentile is {glass_ceiling_index:.1f}x")
    print(f"      larger than at the 10th percentile.")
    print(f"      → Women face stronger barriers at the top of the distribution.")
elif glass_ceiling_index < 0.9:
    print(f"   🟡 STICKY FLOOR DETECTED")
    print(f"      The gender gap at the 10th percentile is {1/glass_ceiling_index:.1f}x")
    print(f"      larger than at the 90th percentile.")
    print(f"      → Women face stronger barriers at the bottom of the distribution.")
else:
    print(f"   ➡️  UNIFORM GAP")
    print(f"      The gender gap is relatively constant across the distribution.")

## 4. Glass Ceiling by Subgroup

In [ ]:
# ============================================================================
# GLASS CEILING BY OCCUPATION
# ============================================================================

noc_labels = {
    0: 'Management', 1: 'Business', 2: 'Sciences',
    3: 'Health', 4: 'Education', 5: 'Arts',
    6: 'Sales/Service', 7: 'Trades', 8: 'Resources', 9: 'Manufacturing'
}

print("Analyzing glass ceiling by occupation...\n")

occ_ceiling = []

for noc in sorted(df_sample['NOC_10'].dropna().unique()):
    if noc not in noc_labels:
        continue
    
    occ_data = df_sample[df_sample['NOC_10'] == noc].copy()
    
    if len(occ_data) < 5000:
        continue
    
    X_occ = occ_data[['IS_FEMALE', 'AGE', 'AGE_SQ', 'EDUC', 'EXP_PROXY']]
    X_occ = sm.add_constant(X_occ)
    y_occ = occ_data['LOG_WAGE']
    
    try:
        # 10th and 90th quantile
        model_10 = QuantReg(y_occ, X_occ).fit(q=0.10, max_iter=MAX_ITER)
        model_90 = QuantReg(y_occ, X_occ).fit(q=0.90, max_iter=MAX_ITER)
        
        gap_10 = abs(model_10.params['IS_FEMALE'])
        gap_90 = abs(model_90.params['IS_FEMALE'])
        gci = gap_90 / gap_10 if gap_10 > 0 else np.nan
        
        occ_ceiling.append({
            'Occupation': noc_labels[noc],
            'NOC': noc,
            'Gap_10th': gap_10,
            'Gap_90th': gap_90,
            'GCI': gci,
            'N': len(occ_data)
        })
        print(f"  {noc_labels[noc]:<20}: GCI = {gci:.2f}")
        
    except Exception as e:
        print(f"  {noc_labels[noc]:<20}: Could not estimate")

occ_ceiling_df = pd.DataFrame(occ_ceiling).sort_values('GCI', ascending=False)

In [ ]:
# ============================================================================
# VISUALIZATION: GCI BY OCCUPATION
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['darkred' if g > 1.2 else 'orange' if g > 1 else 'green' for g in occ_ceiling_df['GCI']]

bars = ax.barh(occ_ceiling_df['Occupation'], occ_ceiling_df['GCI'], color=colors, edgecolor='black')
ax.axvline(x=1.0, color='black', linestyle='--', linewidth=2, label='No ceiling (GCI=1)')

ax.set_xlabel('Glass Ceiling Index (Gap at 90th / Gap at 10th)')
ax.set_title('Glass Ceiling Index by Occupation\n(GCI > 1 = Glass Ceiling, GCI < 1 = Sticky Floor)')
ax.legend()

# Add value labels
for bar, gci in zip(bars, occ_ceiling_df['GCI']):
    width = bar.get_width()
    ax.annotate(f'{gci:.2f}',
                xy=(width, bar.get_y() + bar.get_height()/2),
                xytext=(5, 0), textcoords="offset points",
                ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/glass_ceiling_by_occupation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("GLASS CEILING BY OCCUPATION")
print("=" * 60)
print(f"\nOccupation with strongest ceiling: {occ_ceiling_df.iloc[0]['Occupation']} (GCI = {occ_ceiling_df.iloc[0]['GCI']:.2f})")
print(f"Occupation with weakest ceiling:   {occ_ceiling_df.iloc[-1]['Occupation']} (GCI = {occ_ceiling_df.iloc[-1]['GCI']:.2f})")

## 5. Counterfactual Distributions

In [ ]:
# ============================================================================
# COUNTERFACTUAL WAGE DISTRIBUTIONS
# ============================================================================

# What would women earn if they were paid like men?

# Separate data
male_data = df_sample[df_sample['IS_FEMALE'] == 0].copy()
female_data = df_sample[df_sample['IS_FEMALE'] == 1].copy()

# Fit OLS on male data
X_male = male_data[['AGE', 'AGE_SQ', 'EDUC', 'EXP_PROXY']]
X_male = sm.add_constant(X_male)
y_male = male_data['LOG_WAGE']

male_model = sm.OLS(y_male, X_male).fit()

# Predict counterfactual wages for females using male coefficients
X_female = female_data[['AGE', 'AGE_SQ', 'EDUC', 'EXP_PROXY']]
X_female = sm.add_constant(X_female)

female_data['CF_LOG_WAGE'] = male_model.predict(X_female)
female_data['CF_WAGE'] = np.exp(female_data['CF_LOG_WAGE'])

# Compare distributions
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(female_data['REAL_HRLYEARN'], bins=50, alpha=0.5, label='Actual Female Wages', 
        color='coral', density=True)
ax.hist(female_data['CF_WAGE'], bins=50, alpha=0.5, label='Counterfactual (if paid like men)', 
        color='green', density=True)
ax.hist(male_data['REAL_HRLYEARN'], bins=50, alpha=0.3, label='Actual Male Wages', 
        color='steelblue', density=True)

ax.axvline(female_data['REAL_HRLYEARN'].median(), color='coral', linestyle='--', linewidth=2)
ax.axvline(female_data['CF_WAGE'].median(), color='green', linestyle='--', linewidth=2)
ax.axvline(male_data['REAL_HRLYEARN'].median(), color='steelblue', linestyle='--', linewidth=2)

ax.set_xlabel('Hourly Wage ($)')
ax.set_ylabel('Density')
ax.set_title('Counterfactual Wage Distribution\n(What would women earn if paid like men?)')
ax.legend()
ax.set_xlim(0, 80)

plt.tight_layout()
plt.savefig('reports/figures/counterfactual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
actual_median = female_data['REAL_HRLYEARN'].median()
cf_median = female_data['CF_WAGE'].median()
wage_gain = cf_median - actual_median

print("\n" + "=" * 60)
print("COUNTERFACTUAL ANALYSIS")
print("=" * 60)
print(f"\n  Actual female median wage:       ${actual_median:.2f}")
print(f"  Counterfactual median wage:      ${cf_median:.2f}")
print(f"  Potential gain from equal pay:   ${wage_gain:.2f} ({wage_gain/actual_median:.1%})")

## 6. Time Trends in Glass Ceiling

In [ ]:
# ============================================================================
# GLASS CEILING OVER TIME
# ============================================================================

print("Estimating glass ceiling index by year...\n")

yearly_gci = []

for year in sorted(df_sample['YEAR'].unique()):
    year_data = df_sample[df_sample['YEAR'] == year].copy()
    
    if len(year_data) < MIN_GROUP_N:
        continue
    
    X_year = year_data[['IS_FEMALE', 'AGE', 'AGE_SQ', 'EDUC', 'EXP_PROXY']]
    X_year = sm.add_constant(X_year)
    y_year = year_data['LOG_WAGE']
    
    try:
        model_10 = QuantReg(y_year, X_year).fit(q=0.10, max_iter=MAX_ITER)
        model_50 = QuantReg(y_year, X_year).fit(q=0.50, max_iter=MAX_ITER)
        model_90 = QuantReg(y_year, X_year).fit(q=0.90, max_iter=MAX_ITER)
        
        gap_10 = abs(model_10.params['IS_FEMALE'])
        gap_50 = abs(model_50.params['IS_FEMALE'])
        gap_90 = abs(model_90.params['IS_FEMALE'])
        gci = gap_90 / gap_10 if gap_10 > 0 else np.nan
        
        yearly_gci.append({
            'Year': year,
            'Gap_10th': gap_10,
            'Gap_50th': gap_50,
            'Gap_90th': gap_90,
            'GCI': gci
        })
        print(f"  {year}: GCI = {gci:.3f} (10th: {gap_10:.3f}, 90th: {gap_90:.3f})")
        
    except:
        print(f"  {year}: Could not estimate")

yearly_gci_df = pd.DataFrame(yearly_gci)

In [ ]:
# ============================================================================
# VISUALIZATION: GCI OVER TIME
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Glass Ceiling Index over time
ax1 = axes[0]
ax1.plot(yearly_gci_df['Year'], yearly_gci_df['GCI'], 'o-', color='purple', linewidth=2, markersize=8)
ax1.axhline(y=1.0, color='black', linestyle='--', linewidth=1, label='No ceiling')
ax1.fill_between(yearly_gci_df['Year'], 1, yearly_gci_df['GCI'], 
                 where=yearly_gci_df['GCI'] > 1, alpha=0.3, color='red', label='Glass ceiling')
ax1.fill_between(yearly_gci_df['Year'], 1, yearly_gci_df['GCI'], 
                 where=yearly_gci_df['GCI'] < 1, alpha=0.3, color='green', label='Sticky floor')
ax1.set_xlabel('Year')
ax1.set_ylabel('Glass Ceiling Index')
ax1.set_title('Glass Ceiling Index Over Time')
ax1.legend()

# Plot 2: Gaps by quantile over time
ax2 = axes[1]
ax2.plot(yearly_gci_df['Year'], yearly_gci_df['Gap_10th'] * 100, 'o-', label='10th percentile', color='blue')
ax2.plot(yearly_gci_df['Year'], yearly_gci_df['Gap_50th'] * 100, 's-', label='50th percentile', color='green')
ax2.plot(yearly_gci_df['Year'], yearly_gci_df['Gap_90th'] * 100, '^-', label='90th percentile', color='red')
ax2.set_xlabel('Year')
ax2.set_ylabel('Gender Gap (%)')
ax2.set_title('Gender Gap by Quantile Over Time')
ax2.legend()

plt.tight_layout()
plt.savefig('reports/figures/glass_ceiling_trends.png', dpi=150, bbox_inches='tight')
plt.show()

# Trend analysis
from scipy.stats import linregress
slope, intercept, r, p, se = linregress(yearly_gci_df['Year'], yearly_gci_df['GCI'])

print("\n" + "=" * 60)
print("GLASS CEILING TREND ANALYSIS")
print("=" * 60)
print(f"\n  Trend in GCI: {slope:.4f} per year (p = {p:.4f})")
if slope > 0 and p < 0.05:
    print("  → Glass ceiling is WORSENING over time ⚠️")
elif slope < 0 and p < 0.05:
    print("  → Glass ceiling is IMPROVING over time ✓")
else:
    print("  → No significant trend detected")

## 7. Summary

In [ ]:
# ============================================================================
# SUMMARY
# ============================================================================

print("=" * 70)
print("GLASS CEILING ANALYSIS - KEY FINDINGS")
print("=" * 70)

print("\n📊 OVERALL PATTERN")
print(f"   Glass Ceiling Index: {glass_ceiling_index:.2f}")
if glass_ceiling_index > 1:
    print(f"   → Gender gap is {(glass_ceiling_index-1)*100:.0f}% LARGER at top than bottom")
    print("   → Evidence of GLASS CEILING")
else:
    print(f"   → Gender gap is {(1-glass_ceiling_index)*100:.0f}% LARGER at bottom than top")
    print("   → Evidence of STICKY FLOOR")

print("\n📈 QUANTILE GAPS")
for _, row in qreg_df.iterrows():
    pct = row['quantile'] * 100
    gap = abs(row['female_coef']) * 100
    print(f"   {pct:.0f}th percentile: {gap:.1f}% gap")

print("\n💼 BY OCCUPATION")
print(f"   Strongest ceiling: {occ_ceiling_df.iloc[0]['Occupation']}")
print(f"   Weakest ceiling:   {occ_ceiling_df.iloc[-1]['Occupation']}")

print("\n⏰ TREND")
print(f"   GCI trend: {slope:.4f}/year (p={p:.3f})")

print("\n" + "=" * 70)
print("POLICY IMPLICATIONS")
print("=" * 70)
print("""
1. The gender gap is not uniform across the wage distribution
   - Different interventions needed at different levels

2. Glass ceiling effects suggest:
   - Barriers to advancement into leadership
   - Need for transparency in promotion decisions
   - Quotas/targets for senior positions may help

3. Sticky floor effects (if present) suggest:
   - Minimum wage increases would disproportionately help women
   - Focus on low-wage sectors with gender concentration

4. Occupational variation:
   - Sector-specific interventions needed
   - Management has particularly strong ceiling effects
""")

# Save results
qreg_df.to_csv('reports/quantile_regression_results.csv', index=False)
occ_ceiling_df.to_csv('reports/glass_ceiling_by_occupation.csv', index=False)
yearly_gci_df.to_csv('reports/glass_ceiling_trends.csv', index=False)

print("\n💾 Results saved to reports/")